In [1]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

In [ ]:
file_input = "/Users/allfakhrii_/Documents/My-Code/PYTHON/punya-mama/template-erdkk-2027 - Sumber Makmur (Final Terhitung).xlsx"
df_final = pd.read_excel(file_input)

In [3]:
musim_tanam_cols = [
    ('MT 1', 'Komoditas Periode Tanam I', 'Luas Lahan (Ha) Periode Tanam I'),
    ('MT 2', 'Komoditas Periode Tanam II', 'Luas Lahan (Ha) Periode Tanam II'),
    ('MT 3', 'Komoditas Periode Tanam III', 'Luas Lahan (Ha) Periode Tanam III')
]

In [4]:
rekap_list = []
for mt_nama, col_kom, col_luas in musim_tanam_cols:
    sub_df = df_final[[col_kom, col_luas]].dropna(subset=[col_kom, col_luas]).copy()
    sub_df[col_kom] = sub_df[col_kom].astype(str).str.strip().str.upper()
    sub_df[col_luas] = pd.to_numeric(sub_df[col_luas], errors='coerce').fillna(0)
    
    grp = sub_df.groupby(col_kom).agg(
        Petani=(col_luas, 'count'),
        Luas=(col_luas, 'sum')
    ).reset_index()
    grp['Musim'] = mt_nama
    grp.rename(columns={col_kom: 'Komoditas'}, inplace=True)
    rekap_list.append(grp)

df_all_rekap = pd.concat(rekap_list, ignore_index=True)

In [5]:
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Rekap Luas Komoditas"

In [6]:
font_title = Font(name="Calibri", size=15, bold=True, color="1B365D")
font_subtitle = Font(name="Calibri", size=10, italic=True, color="4A5568")
font_sec_hdr = Font(name="Calibri", size=12, bold=True, color="1B365D")
font_th = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
font_data = Font(name="Calibri", size=11)
font_total = Font(name="Calibri", size=11, bold=True, color="1B365D")

In [7]:
fill_th = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
fill_zebra = PatternFill(start_color="F7FAFC", end_color="F7FAFC", fill_type="solid")
fill_total = PatternFill(start_color="E2E8F0", end_color="E2E8F0", fill_type="solid")

In [8]:
thin_border = Border(
    left=Side(style='thin', color='CBD5E0'), right=Side(style='thin', color='CBD5E0'),
    top=Side(style='thin', color='CBD5E0'), bottom=Side(style='thin', color='CBD5E0')
)
total_border = Border(
    left=Side(style='thin', color='CBD5E0'), right=Side(style='thin', color='CBD5E0'),
    top=Side(style='thin', color='1B365D'), bottom=Side(style='double', color='1B365D')
)

In [9]:
ws['B2'] = "REKAPITULASI LUAS LAHAN TANAM BERDASARKAN KOMODITAS & MUSIM TANAM"
ws['B2'].font = font_title
ws['B3'] = "Kelompok Tani: SUMBER JAYA | Desa Candinata, Kec. Kutasari | Alokasi e-RDKK 2027"
ws['B3'].font = font_subtitle

In [10]:
ws['B5'] = "Tabel Matriks Total Luas Lahan (Hektar / Ha)"
ws['B5'].font = font_sec_hdr

In [11]:
headers_t1 = ["No", "Komoditas", "MT 1 (Ha)", "MT 2 (Ha)", "MT 3 (Ha)", "Total Setahun (Ha)"]
for col_idx, h_text in enumerate(headers_t1, start=2):
    cell = ws.cell(row=6, column=col_idx, value=h_text)
    cell.font = font_th
    cell.fill = fill_th
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = thin_border
ws.row_dimensions[6].height = 25

In [12]:
commodities = sorted(df_all_rekap['Komoditas'].unique())
pivot_luas = {kom: {'MT 1': 0.0, 'MT 2': 0.0, 'MT 3': 0.0} for kom in commodities}
for _, r in df_all_rekap.iterrows():
    pivot_luas[r['Komoditas']][r['Musim']] = r['Luas']

In [13]:
row_idx = 7
for idx, kom in enumerate(commodities, start=1):
    ws.cell(row=row_idx, column=2, value=idx).alignment = Alignment(horizontal='center')
    ws.cell(row=row_idx, column=3, value=kom).alignment = Alignment(horizontal='left')
    
    ws.cell(row=row_idx, column=4, value=pivot_luas[kom]['MT 1']).number_format = '#,##0.000'
    ws.cell(row=row_idx, column=5, value=pivot_luas[kom]['MT 2']).number_format = '#,##0.000'
    ws.cell(row=row_idx, column=6, value=pivot_luas[kom]['MT 3']).number_format = '#,##0.000'
    ws.cell(row=row_idx, column=7, value=f"=SUM(D{row_idx}:F{row_idx})").number_format = '#,##0.000'
    
    for c_i in range(2, 8):
        c = ws.cell(row=row_idx, column=c_i)
        c.font = font_data
        c.border = thin_border
        if c_i >= 4:
            c.alignment = Alignment(horizontal='right')
        if idx % 2 == 0:
            c.fill = fill_zebra
    row_idx += 1

In [14]:
ws.cell(row=row_idx, column=2, value="")
ws.cell(row=row_idx, column=3, value="TOTAL KESELURUHAN")
ws.cell(row=row_idx, column=4, value=f"=SUM(D7:D{row_idx-1})").number_format = '#,##0.000'
ws.cell(row=row_idx, column=5, value=f"=SUM(E7:E{row_idx-1})").number_format = '#,##0.000'
ws.cell(row=row_idx, column=6, value=f"=SUM(F7:F{row_idx-1})").number_format = '#,##0.000'
ws.cell(row=row_idx, column=7, value=f"=SUM(G7:G{row_idx-1})").number_format = '#,##0.000'

In [15]:
for c_i in range(2, 8):
    c = ws.cell(row=row_idx, column=c_i)
    c.font = font_total
    c.fill = fill_total
    c.border = total_border
    if c_i >= 4:
        c.alignment = Alignment(horizontal='right')

In [16]:
ws.column_dimensions['A'].width = 3
ws.column_dimensions['B'].width = 8
ws.column_dimensions['C'].width = 22
ws.column_dimensions['D'].width = 18
ws.column_dimensions['E'].width = 18
ws.column_dimensions['F'].width = 18
ws.column_dimensions['G'].width = 22

In [18]:
output_rekap = "Rekapitulasi_Luas_Lahan_Musim_Tanam_Sumber_Rejeki.xlsx"
wb.save(output_rekap)
print(f"File Excel rekapitulasi berhasil dibuat: {output_rekap}")

File Excel rekapitulasi berhasil dibuat: Rekapitulasi_Luas_Lahan_Musim_Tanam_Sumber_Rejeki.xlsx
